## TEST AOSS INDEX

In [4]:
session

Session(region_name='us-east-1')

In [3]:
import boto3

# 1. Verificar qué role estás usando
sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f"Role ARN: {identity['Arn']}")

# 2. Verificar si tienes permisos IAM para AOSS
aoss_client = boto3.client('opensearchserverless')
try:
    collections = aoss_client.list_collections()
    print(f"Collections: {collections['collectionSummaries']}")
except Exception as e:
    print(f"Error listando collections: {e}")


Role ARN: arn:aws:sts::697682206292:assumed-role/sgmkr-notebook-tfm-hymm-rec-ml-iar-dev/SageMaker
Collections: [{'id': 'f3cprbwt2yb9t7xvm84a', 'name': 'hymmrec-items-vectors', 'status': 'ACTIVE', 'arn': 'arn:aws:aoss:us-east-1:697682206292:collection/f3cprbwt2yb9t7xvm84a', 'kmsKeyArn': 'auto', 'collectionGroupName': 'hymmrec-vectors-cg'}]


In [5]:
pip install opensearch-py requests-aws4auth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [opensearch-py]0m [opensearch-py]
Note: you may need to restart the kernel to use updated packages.


In [6]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import boto3

region = "us-east-1"
service = "aoss"

credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    service,
    session_token=credentials.token
)

host = "f3cprbwt2yb9t7xvm84a.us-east-1.aoss.amazonaws.com"

client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

In [27]:
# Saltar client.info() — no existe en AOSS
# Ir directo a crear el índice
index_body = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 512
        }
    },
    "mappings": {
        "properties": {
            "item_idx": {"type": "integer"},
            "movie_id": {"type": "integer"},
            "title": {"type": "text"},
            "genres": {"type": "keyword"},
            "synopsis": {"type": "text"},
            "release_year": {"type": "integer"},
            "poster_path": {"type": "keyword"},
            "director": {"type": "keyword"},
            "item_embedding": {
                "type": "knn_vector",
                "dimension": 64,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "parameters": {
                        "ef_construction": 512,
                        "m": 16
                    }
                }
            },
            "attention_weights": {
                "type": "object",
                "properties": {
                    "category": {"type": "float"},
                    "text": {"type": "float"},
                    "image": {"type": "float"}
                }
            },
            "indexed_at": {"type": "date"}
        }
    }
}

try:
    response = client.indices.create(index="hymmrec-items-vectors", body=index_body)
    print("Índice creado:", response)
except Exception as e:
    print(f"Error: {e}")


Índice creado: {'acknowledged': True, 'shards_acknowledged': False, 'index': 'hymmrec-items-vectors'}


In [7]:
client.indices.delete(index="hymmrec-items-vectors")


{'acknowledged': True}

In [8]:
count = client.count(index="hymmrec-items-vectors")
print(f"Documentos en índice: {count['count']}")


Documentos en índice: 9518


In [4]:
# Eliminar índice
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import boto3

region = "us-east-1"
service = "aoss"

credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    service,
    session_token=credentials.token
)

host = "f3cprbwt2yb9t7xvm84a.us-east-1.aoss.amazonaws.com"

client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)
response = client.indices.delete(index="hymmrec-items-vectors")
print("Índice eliminado:", response)


Índice eliminado: {'acknowledged': True}


In [7]:
# 1. Contar documentos indexados
count = client.count(index="hymmrec-items-vectors")
print(f"Total documentos indexados: {count['count']}")

# 2. Ver un documento de ejemplo
sample = client.search(
    index="hymmrec-items-vectors",
    body={"size": 1, "query": {"match_all": {}}}
)
doc = sample['hits']['hits'][0]['_source']
print(f"\nEjemplo documento:")
print(f"  item_idx: {doc['item_idx']}")
print(f"  movie_id: {doc['movie_id']}")
print(f"  title: {doc['title']}")
print(f"  genres_text: {doc['genres_text']}")
print(f"  release_year: {doc['release_year']}")
#print(f"  item_embedding dim: {len(doc['item_embedding'])}")
print(f"  attention_weights: {doc['attention_weights']}")
print(f"  genres_multihot len: {len(doc['genres_multihot'])}")
print(f"  text_emb len: {len(doc['text_emb'])}")
print(f"  img_emb len: {len(doc['img_emb'])}")

# 3. Probar kNN search con un vector random (simula user_embedding)
import numpy as np
fake_user_emb = np.random.randn(128).tolist()

knn_results = client.search(
    index="hymmrec-items-vectors",
    body={
        "size": 5,
        "query": {
            "knn": {
                "item_embedding": {
                    "vector": fake_user_emb,
                    "k": 5
                }
            }
        }
    }
)
print(f"\nkNN search (top-5 con vector random):")
for hit in knn_results['hits']['hits']:
    src = hit['_source']
    print(f"  score={hit['_score']:.4f} | {src['title']} ({src['release_year']}) | movie_id={src['movie_id']}")


Total documentos indexados: 5375

Ejemplo documento:
  item_idx: 1064
  movie_id: 1388
  title: Jaws 2
  genres_text: ['Horror', 'Thriller']
  release_year: 1978
  attention_weights: {'image': 0.09391145408153534, 'text': 0.8509793281555176, 'category': 0.05510919168591499}
  genres_multihot len: 20
  text_emb len: 1024
  img_emb len: 1024

kNN search (top-5 con vector random):
  score=0.6038 | North by Northwest (1959) | movie_id=908
  score=0.6023 | The Insider (1999) | movie_id=3006
  score=0.6007 | My Bodyguard (1980) | movie_id=2240
  score=0.6003 | Blade Runner (1982) | movie_id=541
  score=0.5986 | The Gift (2000) | movie_id=4020


In [11]:
# Buscar los parquets reales
bucket = "hymmrec-dilkehousegold01"
prefix = "data/ml_feature_store/inference/"

paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter='/'):
    for p in page.get('CommonPrefixes', []):
        print(p['Prefix'])


data/ml_feature_store/inference/batch-transform-input/
data/ml_feature_store/inference/batch-transform-output/


In [12]:
import boto3, json, pandas as pd

s3 = boto3.client('s3')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name='us-east-1')

TEST_USER_IDX = 2
FULL_MODEL_ENDPOINT = "hymmrec-full-model-sgmkr-ep"

# Leer el JSONL de batch transform input (tiene item_idx, movie_id, genres_multihot, text_emb, img_emb)
obj = s3.get_object(
    Bucket="hymmrec-dilkehousegold01",
    Key="data/ml_feature_store/inference/batch-transform-input/items_for_batch.jsonl"
)
lines = obj['Body'].read().decode().strip().split('\n')
items = [json.loads(line) for line in lines]
print(f"Items: {len(items)}")

# Brute force: invocar full model para todos los items
results = []
for idx, item in enumerate(items):
    try:
        payload = json.dumps({
            "user_idx": TEST_USER_IDX,
            "item_idx": item['item_idx'],
            "genres_multihot": item['genres_multihot'],
            "text_emb": item['text_emb'],
            "img_emb": item['img_emb']
        })
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=FULL_MODEL_ENDPOINT,
            ContentType="application/json",
            Body=payload
        )
        result = json.loads(response['Body'].read())
        result['item_idx'] = item['item_idx']
        result['movie_id'] = item['movie_id']
        results.append(result)
    except Exception as e:
        if idx < 3:
            print(f"Error item {idx}: {e}")

    if (idx + 1) % 500 == 0:
        print(f"Progreso: {idx+1}/{len(items)}")

# TOP-10 brute force
df_results = pd.DataFrame(results)
print(f"\nTotal scored: {len(df_results)}")
print("\n=== TOP-10 BRUTE FORCE por pred_rating_stars ===")
print(df_results.nlargest(10, 'pred_rating_stars')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction']].to_string())

print("\n=== BOTTOM-10 (peores) ===")
print(df_results.nsmallest(10, 'pred_rating_stars')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction']].to_string())

print(f"\n=== Estadísticas ===")
print(df_results['pred_rating_stars'].describe())


Items: 5375
Progreso: 500/5375
Progreso: 1000/5375
Progreso: 1500/5375
Progreso: 2000/5375
Progreso: 2500/5375
Progreso: 3000/5375
Progreso: 3500/5375
Progreso: 4000/5375
Progreso: 4500/5375
Progreso: 5000/5375

Total scored: 5375

=== TOP-10 BRUTE FORCE por pred_rating_stars ===
      item_idx  movie_id  pred_rating_stars  prob_interaction
106         97       110           4.458040          0.795869
973       6669     58998           4.415615          0.102946
1095       973      1278           4.415014          0.184259
30         907      1210           4.408638          0.403590
3523      6460     53894           4.401769          0.014774
1827      2140      2858           4.361008          0.647017
1108      3350      4571           4.359001          0.032994
1968       922      1225           4.353201          0.060549
1931      1730      2329           4.350374          0.303252
2871      7022     70286           4.350012          0.240323

=== BOTTOM-10 (peores) ===
      ite

In [13]:
print("\n=== TOP-10 BRUTE FORCE por hybrid_score ===")
print(df_results.nlargest(10, 'hybrid_score')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction', 'hybrid_score']].to_string())



=== TOP-10 BRUTE FORCE por hybrid_score ===
      item_idx  movie_id  pred_rating_stars  prob_interaction  hybrid_score
106         97       110           4.458040          0.795869      0.688037
146         32        34           4.017374          0.809430      0.610588
1138      1214      1617           4.273091          0.728167      0.595839
1010       277       318           4.283348          0.719711      0.590765
1916         1         2           3.983773          0.791832      0.590662
1842        10        11           3.955696          0.773377      0.571466
3171       589       728           3.897741          0.764459      0.553801
1827      2140      2858           4.361008          0.647017      0.543657
2875       260       300           3.661900          0.816465      0.543337
1847       224       260           4.133087          0.689327      0.539930


In [14]:
import boto3, json, pandas as pd

s3 = boto3.client('s3')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name='us-east-1')

TEST_USER_IDX = 200
FULL_MODEL_ENDPOINT = "hymmrec-full-model-sgmkr-ep"

# Leer el JSONL de batch transform input (tiene item_idx, movie_id, genres_multihot, text_emb, img_emb)
obj = s3.get_object(
    Bucket="hymmrec-dilkehousegold01",
    Key="data/ml_feature_store/inference/batch-transform-input/items_for_batch.jsonl"
)
lines = obj['Body'].read().decode().strip().split('\n')
items = [json.loads(line) for line in lines]
print(f"Items: {len(items)}")

# Brute force: invocar full model para todos los items
results = []
for idx, item in enumerate(items):
    try:
        payload = json.dumps({
            "user_idx": TEST_USER_IDX,
            "item_idx": item['item_idx'],
            "genres_multihot": item['genres_multihot'],
            "text_emb": item['text_emb'],
            "img_emb": item['img_emb']
        })
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=FULL_MODEL_ENDPOINT,
            ContentType="application/json",
            Body=payload
        )
        result = json.loads(response['Body'].read())
        result['item_idx'] = item['item_idx']
        result['movie_id'] = item['movie_id']
        results.append(result)
    except Exception as e:
        if idx < 3:
            print(f"Error item {idx}: {e}")

    if (idx + 1) % 500 == 0:
        print(f"Progreso: {idx+1}/{len(items)}")

# TOP-10 brute force
df_results = pd.DataFrame(results)
print(f"\nTotal scored: {len(df_results)}")
print("\n=== TOP-10 BRUTE FORCE por pred_rating_stars ===")
print(df_results.nlargest(10, 'pred_rating_stars')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction']].to_string())

print("\n=== BOTTOM-10 (peores) ===")
print(df_results.nsmallest(10, 'pred_rating_stars')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction']].to_string())

print(f"\n=== Estadísticas ===")
print(df_results['pred_rating_stars'].describe())

Items: 5375
Progreso: 500/5375
Progreso: 1000/5375
Progreso: 1500/5375
Progreso: 2000/5375
Progreso: 2500/5375
Progreso: 3000/5375
Progreso: 3500/5375
Progreso: 4000/5375
Progreso: 4500/5375
Progreso: 5000/5375

Total scored: 5375

=== TOP-10 BRUTE FORCE por pred_rating_stars ===
      item_idx  movie_id  pred_rating_stars  prob_interaction
30         907      1210           4.835725          0.886203
1095       973      1278           4.803877          0.416655
987        905      1208           4.776653          0.373876
1983       918      1221           4.759179          0.678880
1963       899      1201           4.745824          0.425704
2929      1207      1610           4.741207          0.354333
1827      2140      2858           4.732318          0.815180
2804       965      1270           4.724627          0.277634
130        954      1259           4.721850          0.426352
3058       920      1223           4.721434          0.180920

=== BOTTOM-10 (peores) ===
      ite

In [15]:
print("\n=== TOP-10 BRUTE FORCE por hybrid_score ===")
print(df_results.nlargest(10, 'hybrid_score')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction', 'hybrid_score']].to_string())



=== TOP-10 BRUTE FORCE por hybrid_score ===
      item_idx  movie_id  pred_rating_stars  prob_interaction  hybrid_score
1010       277       318           4.694156          0.970589      0.896377
1051       509       593           4.696012          0.927446      0.856963
30         907      1210           4.835725          0.886203      0.849807
1916         1         2           4.437483          0.979644      0.841877
1847       224       260           4.578480          0.931944      0.833736
106         97       110           4.710665          0.894146      0.829469
164         31        32           4.526674          0.909398      0.801788
1839       417       480           4.589956          0.891986      0.800547
65         506       589           4.653229          0.872122      0.796515
140        257       296           4.350942          0.932065      0.780824


In [1]:
import boto3, json, pandas as pd

s3 = boto3.client('s3')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name='us-east-1')

TEST_USER_IDX = 189
FULL_MODEL_ENDPOINT = "hymmrec-full-model-sgmkr-ep"

# Leer el JSONL de batch transform input (tiene item_idx, movie_id, genres_multihot, text_emb, img_emb)
obj = s3.get_object(
    Bucket="hymmrec-dilkehousegold01",
    Key="data/ml_feature_store/inference/batch-transform-input/items_for_batch.jsonl"
)
lines = obj['Body'].read().decode().strip().split('\n')
items = [json.loads(line) for line in lines]
print(f"Items: {len(items)}")

# Brute force: invocar full model para todos los items
results = []
for idx, item in enumerate(items):
    try:
        payload = json.dumps({
            "user_idx": TEST_USER_IDX,
            "item_idx": item['item_idx'],
            "genres_multihot": item['genres_multihot'],
            "text_emb": item['text_emb'],
            "img_emb": item['img_emb']
        })
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName=FULL_MODEL_ENDPOINT,
            ContentType="application/json",
            Body=payload
        )
        result = json.loads(response['Body'].read())
        result['item_idx'] = item['item_idx']
        result['movie_id'] = item['movie_id']
        results.append(result)
    except Exception as e:
        if idx < 3:
            print(f"Error item {idx}: {e}")

    if (idx + 1) % 500 == 0:
        print(f"Progreso: {idx+1}/{len(items)}")

# TOP-10 brute force
df_results = pd.DataFrame(results)
print(f"\nTotal scored: {len(df_results)}")
print("\n=== TOP-10 BRUTE FORCE por pred_rating_stars ===")
print(df_results.nlargest(10, 'pred_rating_stars')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction']].to_string())

print("\n=== BOTTOM-10 (peores) ===")
print(df_results.nsmallest(10, 'pred_rating_stars')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction']].to_string())

print(f"\n=== Estadísticas ===")
print(df_results['pred_rating_stars'].describe())

Items: 5375
Progreso: 500/5375
Progreso: 1000/5375
Progreso: 1500/5375
Progreso: 2000/5375
Progreso: 2500/5375
Progreso: 3000/5375
Progreso: 3500/5375
Progreso: 4000/5375
Progreso: 4500/5375
Progreso: 5000/5375

Total scored: 5375

=== TOP-10 BRUTE FORCE por pred_rating_stars ===
      item_idx  movie_id  pred_rating_stars  prob_interaction
106         97       110           4.326088          0.817337
948        123       150           4.256949          0.863313
1825      3609      4973           4.174122          0.418530
1910      6281     48780           4.167421          0.817922
30         907      1210           4.162860          0.518231
1051       509       593           4.155272          0.594581
2871      7022     70286           4.134390          0.392061
2902      1280      1704           4.113229          0.370439
2892      8577    122882           4.109991          0.543361
69        5838     33166           4.103320          0.225481

=== BOTTOM-10 (peores) ===
      ite

In [4]:
print("\n=== TOP-10 BRUTE FORCE por hybrid_score ===")
print(df_results.nlargest(10, 'hybrid_score')[['item_idx', 'movie_id', 'pred_rating_stars', 'prob_interaction', 'hybrid_score']].to_string())



=== TOP-10 BRUTE FORCE por hybrid_score ===
      item_idx  movie_id  pred_rating_stars  prob_interaction  hybrid_score
948        123       150           4.256949          0.863313      0.702942
106         97       110           4.326088          0.817337      0.679634
1910      6281     48780           4.167421          0.817922      0.647676
172       3160      4270           4.022359          0.781965      0.590845
2929      1207      1610           4.068979          0.765668      0.587455
2887      4890      7361           3.769320          0.787373      0.545122
2779      4344      6377           3.638058          0.770100      0.507892
1010       277       318           4.087413          0.652259      0.503448
1916         1         2           3.733567          0.726977      0.496810
1141      6370     51255           4.003868          0.660088      0.495704


In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import boto3
import json

# Setup
region = "us-east-1"
host = "f3cprbwt2yb9t7xvm84a.us-east-1.aoss.amazonaws.com"
index_name = "hymmrec-items-vectors"

credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    "aoss",
    session_token=credentials.token
)

client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

# ----- BÚSQUEDA VECTORIAL -----
# Pega aquí el item_embedding (128D) que obtuviste de Athena
item_embedding = [...]  # tu vector de 128 floats

# kNN search: top-20 ítems más similares
results = client.search(
    index=index_name,
    body={
        "size": 20,
        "query": {
            "knn": {
                "item_embedding": {
                    "vector": item_embedding,
                    "k": 20
                }
            }
        },
        "_source": ["item_idx", "movie_id", "title", "genres_text", "release_year"]
    }
)

# Imprimir resultados
print(f"{'Rank':<5} {'Score':<10} {'Movie ID':<10} {'Title':<40} {'Genres'}")
print("-" * 100)
for i, hit in enumerate(results['hits']['hits'], 1):
    src = hit['_source']
    print(f"{i:<5} {hit['_score']:<10.4f} {src['movie_id']:<10} {src['title']:<40} {src.get('genres_text', '')}")


In [8]:
!pip install pyarrow


  Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl (48.8 MB)


In [3]:
import boto3, pickle, io
s3 = boto3.client('s3')
obj = s3.get_object(Bucket='hymmrec-sagemaker-assets', Key='hymmrec/model_artefacts/embeddings/embeddings_catalog.pkl')
catalog = pickle.loads(obj['Body'].read())
print(f"Items en embeddings_catalog: {len(catalog)}")

Items en embeddings_catalog: 9594


/tmp/ipykernel_17416/2518796187.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  catalog = pickle.loads(obj['Body'].read())


In [4]:
sample_keys = sorted(list(catalog.keys()))[:10]
max_key = max(catalog.keys())
print(f"Primeras keys: {sample_keys}")
print(f"Max key: {max_key}")


Primeras keys: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Max key: 193609
